# nb10: BWDF Geometric Proxy

**IMPORTANT — Methodological caveat (read this first):**

The "real" Bond-Weighted Distribution Function (BWDF) is defined by Belli, Zurek, Errea (2025) and implemented in `lobsterpy`. It uses **ICOHP / ICOBI / ICOOP** values from a LOBSTER calculation as the bond-strength weight:

$$\text{BWDF}(r) = \sum_{B>A} \delta(r - |\mathbf{r}_{AB}|) \times B_{AB}$$

where $B_{AB}$ is the bond strength from LOBSTER (an integral of the COHP up to the Fermi level).

**We do not have LOBSTER outputs (only POSCAR + INCAR + vasprun.xml).** Running LOBSTER for 99 compounds requires re-running VASP with `LWAVE=.TRUE.`, then a separate LOBSTER calculation. That is days of compute.

**This notebook implements a geometric proxy, NOT the LobsterPy BWDF.** We replace the bond strength weight $B_{AB}$ with a chemistry-motivated proxy:

$$B_{AB}^{\text{proxy}} = |\chi_A - \chi_B|$$

(absolute electronegativity difference — a textbook proxy for ionicity/bond strength). We also try `(Z_A * Z_B)` (heavy-atom bond proxy), which is closer to how SOC strength scales.

**Why this is still useful:**
1. It encodes the *shape* of the bond-distance distribution weighted by chemistry.
2. The asymmetry index (ASI) we compute from it is a real geometric quantity.
3. If it improves over C6, we have evidence that the bonding distribution carries Rashba-relevant signal — which then justifies running real LOBSTER on a few compounds.

**When you talk to Prof. Bhattacharya:** be explicit that this is a proxy. Do not let her think you ran real BWDF.

---

**What this notebook does:**

1. Load merged dataframe (same as nb7) and collapse to 99 compounds.
2. For each compound, parse POSCAR_std, find all bonds within a cutoff using pymatgen `CrystalNN` (or simple distance cutoff fallback), compute pairwise distances + bond weights.
3. Build a **histogram of weighted bond distances** (BWDF). Compute summary statistics (mean, std, skew, kurtosis, weighted mean).
4. Compute an **asymmetry index (ASI)** per atom and aggregate over the structure.
5. Add these features to df_99_ext, run the same C6 + 1 evaluation as nb9.


## Cell 1: Imports & paths

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from time import time

from pymatgen.core import Structure, Element
from pymatgen.analysis.local_env import CrystalNN

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
from scipy.stats import skew, kurtosis

warnings.filterwarnings('ignore')

# ============================================================================
# PATHS  (notebook lives in Keshav-DDP/new-descriptors/)
# ============================================================================
BASE_DIR = os.path.abspath(os.path.join('..'))   # -> Keshav-DDP/

OLD_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors_old.csv')
NEW_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors.csv')

# POSCAR_std files: Keshav-DDP/Inverse-design/rashba/{uid}/POSCAR_std
POSCAR_DIR = os.path.join(BASE_DIR, 'Inverse-design', 'rashba')

RESULTS_DIR = os.path.join('.', 'nb10_bwdf-results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('OLD_CSV   :', OLD_CSV,   '  exists:', os.path.exists(OLD_CSV))
print('NEW_CSV   :', NEW_CSV,   '  exists:', os.path.exists(NEW_CSV))
print('POSCAR_DIR:', POSCAR_DIR, '  exists:', os.path.exists(POSCAR_DIR))
print('RESULTS   :', RESULTS_DIR)


## Cell 2: Load merged data, collapse to 99 compounds, reproduce baseline

Same as nb9. If baseline R² differs from 0.643 by more than 0.05, stop and debug.

In [ ]:
df_old = pd.read_csv(OLD_CSV)
df_new = pd.read_csv(NEW_CSV)

ID_COLS = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
TARGET = 'Rashba_parameter'

df_merged = df_old[ID_COLS].copy()
old_features = [c for c in df_old.columns if c not in ID_COLS]
new_features = [c for c in df_new.columns if c not in ID_COLS]
overlap = set(old_features) & set(new_features)

for col in old_features:
    name = f'old_{col}' if col in overlap else col
    df_merged[name] = df_old[col].values
for col in new_features:
    name = f'new_{col}' if col in overlap else col
    df_merged[name] = df_new[col].values

idx_max = df_merged.groupby('uid')[TARGET].idxmax()
df_99 = df_merged.loc[idx_max].reset_index(drop=True)
y_99 = df_99[TARGET].values
print(f'99-row df: {df_99.shape[0]} compounds')

XGB_REG_PARAMS = dict(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=1.0,
    random_state=42, verbosity=0,
)

BASELINE_RAW = ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean',
                'pmid_afs_gauss_std', 'kpath_angle_deg', 'ehull']


def resolve_features(feat_list, df_cols):
    cols = set(df_cols)
    out = []
    missing = []
    for f in feat_list:
        if f in cols: out.append(f)
        elif f'old_{f}' in cols: out.append(f'old_{f}')
        elif f'new_{f}' in cols: out.append(f'new_{f}')
        else: missing.append(f)
    if missing: print(f'  WARNING missing: {missing}')
    return out


def eval_reg_99(features, df, y):
    X = df[features].fillna(0).values
    model = XGBRegressor(**XGB_REG_PARAMS)
    y_pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
    return r2_score(y, y_pred), mean_absolute_error(y, y_pred)


BASELINE = resolve_features(BASELINE_RAW, df_99.columns)
r2_base, mae_base = eval_reg_99(BASELINE, df_99, y_99)
print(f'Baseline LOO R2 = {r2_base:.4f}   MAE = {mae_base:.4f}')

if abs(r2_base - 0.643) > 0.05:
    print('  WARNING: baseline differs from nb7 by > 0.05. Investigate.')


## Cell 3: BWDF computation

For each compound:
1. Load POSCAR_std (the corrected POSCAR with elements on line 1 — see DDP context).
2. Find all atom pairs within `R_CUTOFF` (default 5 Å) using a simple distance loop. Periodic images are accounted for via pymatgen's `get_all_neighbors`.
3. For each pair, compute distance $r_{AB}$ and a bond-weight proxy $B_{AB}$.
4. Build histogram $H(r) = \sum_{AB} B_{AB} \cdot \delta(r - r_{AB})$ on a fine grid.
5. Extract summary statistics from $H(r)$.

**Two weight schemes:**
- `bw_chi`: $|\chi_A - \chi_B|$ (electronegativity difference, ionicity proxy)
- `bw_z`:   $Z_A \cdot Z_B$ (heavy-atom bond, SOC-relevant)

We compute features under both and let XGBoost decide.

**Speed:** ~99 compounds × ~50 ms structure load × ~20 atoms × neighbor-finding = roughly 30-60 seconds total. Progress bar prints every 10 compounds.

In [ ]:
# BWDF parameters
R_CUTOFF = 5.0       # Angstroms; max bond distance to consider
N_BINS = 100         # number of distance bins for histogram
R_GRID = np.linspace(0.5, R_CUTOFF, N_BINS)
DR = R_GRID[1] - R_GRID[0]


def load_structure(uid):
    """Load POSCAR_std for a given uid. Returns Structure or None."""
    p = os.path.join(POSCAR_DIR, uid, 'POSCAR_std')
    if not os.path.exists(p):
        return None
    try:
        return Structure.from_file(p)
    except Exception as e:
        print(f'    Failed to load {p}: {e}')
        return None


def bond_weight_chi(z1, z2):
    """|chi_A - chi_B| using Pauling electronegativity."""
    el1 = Element.from_Z(z1)
    el2 = Element.from_Z(z2)
    chi1 = el1.X if el1.X is not None else 0.0
    chi2 = el2.X if el2.X is not None else 0.0
    return abs(chi1 - chi2)


def bond_weight_z(z1, z2):
    """Z_A * Z_B (heavy-atom bond proxy)."""
    return float(z1) * float(z2)


def compute_bwdf_for_structure(struct, weight_func, r_cutoff=R_CUTOFF, sigma=0.1):
    """Return Gaussian-broadened BWDF on R_GRID, plus list of (r, w) pairs."""
    # get_all_neighbors handles periodic boundaries; returns list-of-list of (site, dist, idx)
    all_neighbors = struct.get_all_neighbors(r=r_cutoff)

    bond_list = []   # list of (r_AB, weight)
    for i, neighbors_of_i in enumerate(all_neighbors):
        z_i = struct.species[i].Z
        for nbr in neighbors_of_i:
            # In pymatgen, neighbor object has .nn_distance and .specie
            d = nbr.nn_distance
            z_j = nbr.specie.Z
            if d < 0.5: continue   # skip self / unphysical
            w = weight_func(z_i, z_j)
            bond_list.append((d, w))

    if not bond_list:
        return np.zeros(N_BINS), []

    # Gaussian-broadened histogram (avoids delta-spike noise)
    bwdf = np.zeros(N_BINS)
    rs = np.array([b[0] for b in bond_list])
    ws = np.array([b[1] for b in bond_list])
    for r, w in zip(rs, ws):
        bwdf += w * np.exp(-((R_GRID - r) ** 2) / (2 * sigma ** 2))
    return bwdf, bond_list


def bwdf_summary_stats(bwdf, prefix):
    """Extract scalar features from a 1D BWDF on R_GRID."""
    if bwdf.sum() == 0:
        return {
            f'{prefix}_total': 0.0,
            f'{prefix}_mean_r': np.nan,
            f'{prefix}_std_r': np.nan,
            f'{prefix}_skew': np.nan,
            f'{prefix}_kurt': np.nan,
            f'{prefix}_max_val': 0.0,
            f'{prefix}_max_r': np.nan,
            f'{prefix}_first_peak_r': np.nan,
        }
    p = bwdf / bwdf.sum()  # normalized as probability
    mean_r = float(np.sum(R_GRID * p))
    var_r = float(np.sum((R_GRID - mean_r) ** 2 * p))
    std_r = float(np.sqrt(var_r))
    if std_r > 0:
        skew_r = float(np.sum(((R_GRID - mean_r) / std_r) ** 3 * p))
        kurt_r = float(np.sum(((R_GRID - mean_r) / std_r) ** 4 * p) - 3.0)
    else:
        skew_r = 0.0; kurt_r = 0.0

    max_idx = int(np.argmax(bwdf))

    # First peak: smallest r where local max above 5% of global max
    threshold = 0.05 * bwdf.max()
    first_peak_r = np.nan
    for i in range(1, len(bwdf) - 1):
        if bwdf[i] > threshold and bwdf[i] > bwdf[i-1] and bwdf[i] > bwdf[i+1]:
            first_peak_r = float(R_GRID[i])
            break

    return {
        f'{prefix}_total': float(bwdf.sum() * DR),       # integral
        f'{prefix}_mean_r': mean_r,
        f'{prefix}_std_r': std_r,
        f'{prefix}_skew': skew_r,
        f'{prefix}_kurt': kurt_r,
        f'{prefix}_max_val': float(bwdf.max()),
        f'{prefix}_max_r': float(R_GRID[max_idx]),
        f'{prefix}_first_peak_r': first_peak_r,
    }


## Cell 4: Asymmetry Index (ASI)

From the LobsterPy docs (Belli, Zurek, Errea 2025):

$$\mathbf{V}_x = \frac{1}{B_x} \sum_{\alpha=1}^{B_x} B_{x\alpha} \, \hat{i}_{x\alpha}$$

where $\hat{i}_{x\alpha}$ is the unit vector from atom $x$ to neighbor $\alpha$, weighted by the bond strength. **For a centrosymmetric environment, $\mathbf{V}_x = 0$. For asymmetric environments, $|\mathbf{V}_x| > 0$.**

This is the **most physically motivated descriptor for Rashba** in this notebook — broken inversion symmetry is exactly what creates Rashba SOC. Even with our weight proxy, the geometric asymmetry is captured.

We compute $|\mathbf{V}_x|$ per atom, then aggregate (mean, std, max) across the structure.

**Special z-component:** for 2D materials, the out-of-plane component $V_{x,z}$ is most relevant (Rashba field is perpendicular to the plane). We compute that separately.

In [ ]:
def compute_asi(struct, weight_func, r_cutoff=R_CUTOFF):
    """Per-atom asymmetry vector V_x and aggregated stats."""
    all_neighbors = struct.get_all_neighbors(r=r_cutoff)
    V_per_atom = []        # |V_x| magnitude per atom
    Vz_per_atom = []       # V_x z-component per atom (signed)

    for i, neighbors_of_i in enumerate(all_neighbors):
        site_i = struct[i]
        z_i = site_i.specie.Z
        if not neighbors_of_i:
            V_per_atom.append(0.0); Vz_per_atom.append(0.0); continue

        weighted_disp = np.zeros(3)
        total_w = 0.0
        for nbr in neighbors_of_i:
            d = nbr.nn_distance
            if d < 0.5: continue
            z_j = nbr.specie.Z
            w = weight_func(z_i, z_j)
            # Unit vector from i to j (using neighbor's coords)
            disp = nbr.coords - site_i.coords
            unit = disp / d
            weighted_disp += w * unit
            total_w += w

        if total_w > 0:
            V = weighted_disp / total_w
        else:
            V = np.zeros(3)
        V_per_atom.append(float(np.linalg.norm(V)))
        Vz_per_atom.append(float(V[2]))   # z is out-of-plane for 2D

    V_per_atom = np.array(V_per_atom)
    Vz_per_atom = np.array(Vz_per_atom)
    return {
        'asi_mean': float(np.mean(V_per_atom)),
        'asi_std': float(np.std(V_per_atom)),
        'asi_max': float(np.max(V_per_atom)),
        'asi_z_mean': float(np.mean(Vz_per_atom)),       # signed: net out-of-plane asymmetry
        'asi_z_abs_mean': float(np.mean(np.abs(Vz_per_atom))),
        'asi_z_max': float(np.max(np.abs(Vz_per_atom))),
    }


## Cell 5: Run BWDF + ASI on all 99 compounds

Time estimate: ~1-2 seconds per compound × 99 = 1-3 minutes total.

In [ ]:
print('=' * 70)
print('  Computing BWDF + ASI for 99 compounds')
print('=' * 70)

results = []
t0 = time()
n_failed = 0

for i, row in df_99.iterrows():
    uid = row['uid']
    formula = row['Formula']
    struct = load_structure(uid)

    feat = {'uid': uid}
    if struct is None:
        n_failed += 1
        results.append(feat)
        continue

    try:
        # BWDF with electronegativity weight
        bwdf_chi, _ = compute_bwdf_for_structure(struct, bond_weight_chi)
        feat.update(bwdf_summary_stats(bwdf_chi, 'bwdf_chi'))

        # BWDF with Z*Z weight
        bwdf_z, _ = compute_bwdf_for_structure(struct, bond_weight_z)
        feat.update(bwdf_summary_stats(bwdf_z, 'bwdf_z'))

        # Asymmetry index with both weights
        asi_chi = compute_asi(struct, bond_weight_chi)
        feat.update({f'asi_chi_{k.replace("asi_", "")}': v for k, v in asi_chi.items()})
        asi_z = compute_asi(struct, bond_weight_z)
        feat.update({f'asi_z_{k.replace("asi_", "")}': v for k, v in asi_z.items()})
    except Exception as e:
        print(f'  [{i+1}/99] {uid} ({formula}): FAILED -- {e}')
        n_failed += 1

    results.append(feat)

    if (i + 1) % 10 == 0 or i == 98:
        print(f'  [{i+1}/99] elapsed {time()-t0:.0f}s')

print(f'\nTotal time: {time()-t0:.0f}s')
print(f'Failed: {n_failed} / 99')

bwdf_df = pd.DataFrame(results)
print(f'BWDF feature columns: {len(bwdf_df.columns) - 1}')   # minus uid
print('Sample columns:', [c for c in bwdf_df.columns[:8] if c != 'uid'])

bwdf_df.to_csv(os.path.join(RESULTS_DIR, 'bwdf_features.csv'), index=False)


## Cell 6: Attach BWDF features to df_99 and run C6 + 1 expansion

In [ ]:
# Merge on uid
df_99_ext = df_99.merge(bwdf_df, on='uid', how='left')
print(f'Extended df_99: {df_99_ext.shape}')

NEW_CANDIDATES = [c for c in bwdf_df.columns if c != 'uid']
print(f'New BWDF candidate features: {len(NEW_CANDIDATES)}')

BASELINE = resolve_features(BASELINE_RAW, df_99_ext.columns)

# Phase A: C6 + 1
print('\n' + '=' * 70)
print('  PHASE A: C6 + 1 (BWDF features)')
print('=' * 70)
phase_a_results = []
for cand in NEW_CANDIDATES:
    feats = BASELINE + [cand]
    try:
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        phase_a_results.append({'candidate': cand, 'r2': r2, 'mae': mae,
                                'delta_r2': r2 - r2_base})
    except Exception as e:
        print(f'  {cand}: FAILED -- {e}')

phase_a_df = pd.DataFrame(phase_a_results).sort_values('delta_r2', ascending=False).reset_index(drop=True)
print('\nTop 15 BWDF candidates (C6 + 1):')
print(phase_a_df.head(15).to_string(index=False))
phase_a_df.to_csv(os.path.join(RESULTS_DIR, 'phase_a_bwdf.csv'), index=False)


## Cell 7: Phase B — C6 + 2 (top 5 BWDF features)

In [ ]:
from itertools import combinations

print('=' * 70)
print('  PHASE B: C6 + 2 (top 5 BWDF features)')
print('=' * 70)
top5 = phase_a_df.head(5)['candidate'].tolist()
print('Top 5:', top5)

phase_b = []
for c1, c2 in combinations(top5, 2):
    r2, mae = eval_reg_99(BASELINE + [c1, c2], df_99_ext, y_99)
    phase_b.append({'cand_1': c1, 'cand_2': c2, 'r2': r2, 'mae': mae,
                    'delta_r2': r2 - r2_base})
phase_b_df = pd.DataFrame(phase_b).sort_values('delta_r2', ascending=False).reset_index(drop=True)
print(phase_b_df.to_string(index=False))
phase_b_df.to_csv(os.path.join(RESULTS_DIR, 'phase_b_bwdf.csv'), index=False)

best_a = phase_a_df.iloc[0]['r2']
best_b = phase_b_df.iloc[0]['r2']
print(f'\nBest C6 + 1 (BWDF): R2 = {best_a:.4f}')
print(f'Best C6 + 2 (BWDF): R2 = {best_b:.4f}')
print(f'Baseline:           R2 = {r2_base:.4f}')


## Cell 8: Cross-feature combinations

If both elemental (nb9) and BWDF (this notebook) features improve over baseline, the question is whether they're capturing the same signal or different signals. To test:

Take the **best 1 elemental feature** from nb9 (load `phase_a_ranking.csv` from `nb9_elemental-results/`) and the **best 1 BWDF feature** from this notebook. Run C6 + elem + bwdf and see if it beats both individual additions.

This cell is **OPTIONAL** — only run after you've executed nb9.

In [ ]:
# Try to load nb9 results
nb9_results = os.path.join('.', 'nb9_elemental-results', 'phase_a_ranking.csv')
if os.path.exists(nb9_results):
    nb9_df = pd.read_csv(nb9_results)
    best_elem = nb9_df.iloc[0]['candidate']
    best_elem_r2 = nb9_df.iloc[0]['r2']
    print(f'Best elemental feature (from nb9): {best_elem}  (R2 = {best_elem_r2:.4f})')

    # We need that column in df_99_ext. If nb9 was run separately, the elem feature
    # won't be in this df. We need to reconstruct it OR re-run nb9's Cell 4-5 here.
    # Simpler: warn the user.
    if best_elem not in df_99_ext.columns:
        print(f'  Note: {best_elem} not in current df_99_ext. Run nb9 Cell 4-5 to construct elem features,')
        print(f'        then merge into this notebook. Skipping cross-combo.')
    else:
        best_bwdf = phase_a_df.iloc[0]['candidate']
        feats = BASELINE + [best_elem, best_bwdf]
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        print(f'\nC6 + best_elem + best_bwdf:')
        print(f'  Features: {feats}')
        print(f'  R2 = {r2:.4f} (baseline {r2_base:.4f}, only_bwdf {best_a:.4f}, only_elem {best_elem_r2:.4f})')
        if r2 > max(best_a, best_elem_r2):
            print('  -> Elemental and BWDF features capture DIFFERENT signals. Combine them.')
        else:
            print('  -> Combining did not help. They may capture overlapping signal.')
else:
    print(f'nb9 results not found at {nb9_results}. Run nb9 first, then re-run this cell.')


## Cell 9: Final summary

Save the best feature set found in this notebook.

In [ ]:
best_a_r2 = phase_a_df.iloc[0]['r2']
best_b_r2 = phase_b_df.iloc[0]['r2']

best_phase = 'A'; best_r2 = best_a_r2; best_extras = [phase_a_df.iloc[0]['candidate']]
if best_b_r2 > best_a_r2:
    best_phase = 'B'; best_r2 = best_b_r2
    best_extras = [phase_b_df.iloc[0]['cand_1'], phase_b_df.iloc[0]['cand_2']]

print('=' * 70)
print('  nb10 BWDF FINAL SUMMARY')
print('=' * 70)
print(f'Baseline (C6):  R2 = {r2_base:.4f}')
print(f'Best Phase A:    R2 = {best_a_r2:.4f}  ({best_a_r2 - r2_base:+.4f})')
print(f'Best Phase B:    R2 = {best_b_r2:.4f}  ({best_b_r2 - r2_base:+.4f})')
print()
print(f'BEST: Phase {best_phase}')
print(f'  Features ({len(BASELINE) + len(best_extras)}):')
for f in BASELINE + best_extras:
    print(f'    - {f}')

summary = pd.DataFrame([{
    'phase': f'Phase {best_phase}',
    'r2': best_r2,
    'delta_r2': best_r2 - r2_base,
    'extras': ' + '.join(best_extras),
}])
summary.to_csv(os.path.join(RESULTS_DIR, 'final_summary.csv'), index=False)


## Notes for next steps

1. **Real BWDF.** If anything in this notebook moves R² noticeably (say > +0.02), that's strong evidence to budget LOBSTER calculations on the 99 compounds — at least on a subset. The real ICOHP-weighted BWDF will likely improve further over this proxy.

2. **2D-aware ASI.** Right now, `asi_z_*` uses Cartesian z, which assumes the 2D plane is the xy plane. This is true for C2DB structures (vacuum is along c). Verify by checking `struct.lattice.matrix` for a few compounds.

3. **Per-element BWDF.** The current BWDF sums over all bond pairs. A more refined version computes BWDF separately per element pair (Bi-Te, Bi-O, etc.) and uses those as separate descriptors. Adds many more features but more interpretable.

4. **Don't forget:** when reporting to Prof. Bhattacharya, this is the **geometric proxy**, not LobsterPy BWDF. Be explicit about it.
